In [1]:
# system imports
import os
import time
import sys
import glob
from tqdm import tqdm
#from pathlib import Path

# data access imports
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u

# display imports
from IPython.display import HTML

# multiprocessing imports
import multiprocessing as mp
from multiprocessing.pool import ThreadPool

# data challenge imports
#import MulensModel
import emcee
#from pyLIMA.pyLIMASS import SourceLensProbabilities
import RTModel

# data analysis/visualization imports
import numpy as np
import matplotlib.pyplot as plt

In [2]:
root_dir = '/home/mldc-ipacpeople-admin/teams/mldc-ipacpeople/'
event_data_me = 'event_data_chris/'
events_dir = root_dir + event_data_me

In [3]:
def read_nature_txt(filename, n_chisquares):
    """
    Read Nature.txt if it's not empty and grab the top n_chisquares models. 
    """
    found = False
    n = 0
    chisqs = []
    model_files = []
    with open(filename, "r") as file:
        for line in file:
            # don't do anything until the chisquare line, which is where FinalModels are listed
            if 'chisquare' in line:
                found = True 
                continue # skip to next line, our search for the chisquare line is done
            
            # only then do we start processing lines
            if found == True:
                #print(line.strip())  # strip() removes trailing newlines

                # grab best N chisquares
                if n < n_chisquares:
                    chisq = line.split()[0]
                    model_file = line.split()[1]
                    chisqs.append(chisq)
                    model_files.append(model_file)
                else: 
                    continue
    
                n += 1
    file.close() 

    return chisqs, model_files

What fraction of favored models are planetary events? Of these, what are "contested" (either LS or BS is also among the favored models for that event)?

In [4]:
n_chisquares = 1 # how many of the best chisquares to take
count_ls_list = []
count_lx_list = []
count_lo_list = []
count_bs_list = []
count_bo_list = []
count_ps_list = []
count_px_list = []
event_ids = []
for f in tqdm(np.sort(glob.glob(events_dir+"/*"))):
    event_id = int(f.removeprefix(events_dir+"event_"))
    if event_id < 300000:
        continue

    event_ids.append(event_id)
    event_dir = events_dir + 'event_' + "{:04d}/".format(event_id)
    event_final_models = glob.glob(event_dir+'FinalModels/*')

    # if there are no FinalModels (ie., if either there was no detection or there was no 
    if len(event_final_models)==0:
        count_ps_list.append(0)
        count_px_list.append(0)
        count_ls_list.append(0)
        count_lx_list.append(0)
        count_lo_list.append(0)
        count_bs_list.append(0)
        count_bo_list.append(0)
        continue
    else:
        chisqs, model_filenames = read_nature_txt(event_dir+'Nature.txt', n_chisquares=len(event_final_models))

    # read N best model_filenames to get inferred parameters
    count_ls = 0
    count_lx = 0
    count_lo = 0
    count_bs = 0
    count_bo = 0
    count_ps = 0
    count_px = 0
    #for j in range(n_chisquares):
    for j in range(len(event_final_models)): 
        chisq_temp = chisqs[j]
        model_filename_temp = model_filenames[j]
        # 1) how many events have LS, LX, or LO in FinalModels/?
        if (model_filename_temp[:2] == 'LS'):
            count_ls += 1
        elif (model_filename_temp[:2] == 'LX'):
            count_lx += 1
        elif (model_filename_temp[:2] == 'LO'):
            count_lo += 1
        elif (model_filename_temp[:2] == 'BS'):
            count_bs += 1
        elif (model_filename_temp[:2] == 'BO'):
            count_bo += 1
        elif (model_filename_temp[:2] == 'PS'):
            count_ps += 1
        elif (model_filename_temp[:2] == 'PX'):
            count_px += 1 

    count_ls_list.append(count_ls)
    count_lx_list.append(count_lx)
    count_lo_list.append(count_lo)
    count_bs_list.append(count_bs)
    count_bo_list.append(count_bo)
    count_ps_list.append(count_ps)
    count_px_list.append(count_px)

final_models_df = pd.DataFrame({'event': event_ids, 'PS': count_ps_list, 'PX': count_px_list, 
                                'LS': count_ls_list, 'LX': count_lx_list, 'LO': count_lo_list,
                                'BS': count_bs_list, 'BO': count_bo_list})       
final_models_df


100%|██████████| 218/218 [00:03<00:00, 65.48it/s] 


,event,PS,PX,LS,LX,LO,BS,BO
0,306458,0,0,1,0,0,0,0
1,306463,0,0,0,0,0,0,0
2,306467,0,0,1,0,0,0,0
3,306479,0,0,2,0,0,0,0
4,306493,0,0,1,0,0,0,0
5,306501,0,0,1,0,0,1,0
6,306504,0,0,0,0,1,0,0
7,306505,0,0,0,0,0,0,0
8,306508,0,0,1,0,0,0,0
9,306514,0,0,2,0,0,0,0


In [57]:
category_list = []
s_list = []
q_list = []
u0_list = []
alpha_list = []
rho_list = []
tE_list = []
t0_list = []
flux_bg_213_list = []
flux_source_213_list = []
flux_bg_146_list = []
flux_source_146_list = []
flux_bg_087_list = []
flux_source_087_list = []
t0_anomaly_list = []
y1_anomaly_list = []
y2_anomaly_list = []
delta_chisq_anomaly_list = []
delta_chisq_list = []
s_err_list = []
q_err_list = []
u0_err_list = []
alpha_err_list = []
rho_err_list = []
tE_err_list = []
t0_err_list = []
flux_bg_213_err_list = []
flux_source_213_err_list = []
flux_bg_146_err_list = []
flux_source_146_err_list = []
flux_bg_087_err_list = []
flux_source_087_err_list = []
for event_id in tqdm(event_ids):
    models_path = events_dir + 'event_' + str(event_id) + '/'
    models = glob.glob(models_path +'FinalModels/*')
    for model in models:
        with open(model, 'r') as file:
            # read first line: parameters
            line = file.readline().split()
            category = model.removeprefix("/home/mldc-ipacpeople-admin/teams/mldc-ipacpeople/"+event_data_me+"event_"+str(event_id)+"/FinalModels/")[:2]
            category_list.append(category)
            s, q, u0, alpha, rho, tE, t0 = line[0], line[1], line[2], line[3], line[4], line[5], line[6]
            flux_bg_213, flux_source_213 = line[7], line[8]
            flux_bg_146, flux_source_146 = line[9], line[10]
            flux_bg_087, flux_source_087 = line[11], line[12]
            t0_anomaly, y1_anomaly, y2_anomaly, delta_chisq_anomaly = line[13], line[14], line[15], line[16]
            delta_chisq = line[17]
            s_list.append(s), q_list.append(q), u0_list.append(u0), alpha_list.append(alpha), rho_list.append(rho), tE_list.append(tE), t0_list.append(t0)
            flux_bg_213_list.append(flux_bg_213), flux_source_213_list.append(flux_source_213)
            flux_bg_146_list.append(flux_bg_146), flux_source_146_list.append(flux_source_146)
            flux_bg_087_list.append(flux_bg_087), flux_source_087_list.append(flux_source_087)
            t0_anomaly_list.append(t0_anomaly), y1_anomaly_list.append(y1_anomaly), y2_anomaly_list.append(y2_anomaly), delta_chisq_anomaly_list.append(delta_chisq_anomaly)
            delta_chisq_list.append(delta_chisq)
            
            # read second line: parameter uncertainties
            line = file.readline().split()
            s_err, q_err, u0_err, alpha_err, rho_err, tE_err, t0_err = line[0], line[1], line[2], line[3], line[4], line[5], line[6]
            flux_bg_213_err, flux_source_213_err = line[7], line[8]
            flux_bg_146_err, flux_source_146_err = line[9], line[10]
            flux_bg_087_err, flux_source_087_err = line[11], line[12]
            s_err_list.append(s_err), q_err_list.append(q_err), u0_err_list.append(u0_err), alpha_err_list.append(alpha_err), rho_err_list.append(rho_err), tE_err_list.append(tE_err), t0_err_list.append(t0_err)
            flux_bg_213_err_list.append(flux_bg_213_err), flux_source_213_err_list.append(flux_source_213_err)
            flux_bg_146_err_list.append(flux_bg_146_err), flux_source_146_err_list.append(flux_source_146_err)
            flux_bg_087_err_list.append(flux_bg_087_err), flux_source_087_err_list.append(flux_source_087_err)

    fasdfasdfasdf


  0%|          | 0/16 [00:00<?, ?it/s]

/home/mldc-ipacpeople-admin/teams/mldc-ipacpeople/event_data_chris/event_306458/FinalModels/LS0231-3.txt
LS


NameError: name 'fasdfasdfasdf' is not defined

In [58]:
results = pd.DataFrame({'s': s_list, 's_err': s_err_list, 'q': q_list, 'q_err': q_err_list, 'u0': u0_list, 'u0_err': u0_err_list, 
                        'alpha': alpha_list, 'alpha_err': alpha_err_list, 'rho': rho_list, 'rho_err': rho_err_list,
                       'tE': tE_list, 'tE_err': tE_err_list, 't0': t0_list, 't0_err': t0_err_list, 
                        'flux_bg_213': flux_bg_213_list, 'flux_bg_213_err': flux_bg_213_err_list, 'flux_source_213': flux_source_213_list, 'flux_source_213_err': flux_source_213_err_list, 
                       'flux_bg_146': flux_bg_146_list, 'flux_bg_146_err': flux_bg_146_err_list, 'flux_source_146': flux_source_146_list, 'flux_source_146_err': flux_source_146_err_list, 
                        'flux_bg_087': flux_bg_087_list, 'flux_bg_087_err': flux_bg_087_err_list, 'flux_source_087': flux_source_087_list, 'flux_source_087_err': flux_source_087_err_list,
                       't0_anomaly': t0_anomaly_list, 'y1_anomaly': y1_anomaly_list, 'y2_anomaly': y2_anomaly_list, 'delta_chisq_anomaly': delta_chisq_anomaly_list,
                       'delta_chisq': delta_chisq_list})
results = results.astype(float)
results['category'] = category_list # add this after converting everything else to floats
results

,s,s_err,q,q_err,u0,u0_err,alpha,alpha_err,rho,rho_err,...,flux_bg_087,flux_bg_087_err,flux_source_087,flux_source_087_err,t0_anomaly,y1_anomaly,y2_anomaly,delta_chisq_anomaly,delta_chisq,category
0,0.674402,0.011123,0.000521,0.000013,0.733317,0.036586,4.577516,0.002354,0.00001,1.922505,...,6.55711,0.232831,4.057253,0.000038,12949.040191,-0.768064,-0.206536,13.636995,48912.455336,LS


In [59]:
from datasets import load_dataset
import os

os.environ["HF_TOKEN"] = "hf_your_token_here" # will replace before pushing; dear teammates: please don't compromise me

token = os.environ.get("HF_TOKEN")
assert token, "Set HF_TOKEN environment variable first"

# con.execute("""
#     CREATE SECRET hf_token (
#         TYPE HUGGINGFACE,
#         PROVIDER credential_chain
#     );
# """)

dataset_name = "RGES-PIT/MachineLearning"

# Load metadata and epochs tables (which fit easily in RAM)
meta_df = load_dataset(dataset_name, "metadata", split="train", token=token).to_pandas()
epochs_df = load_dataset(dataset_name, "epochs", split="train", token=token).to_pandas()

print(f"Metadata     : {meta_df.shape[0]:,} rows")
print(f"Epochs       : {epochs_df.shape[0]:,} rows")

Metadata     : 377,999 rows
Epochs       : 98,974 rows


In [63]:
meta_df.loc[meta_df['event_id'].isin(event_ids)][['name','event_id','sim_label','t0lens1','tE_ref','u0lens1',
                                                  'rho','alpha','piE','piEE','piEN','thetaE','ra_deg','dec_deg',
                                                  'Planet_q','Planet_s','Planet_semimajoraxis','Lens_Mass',
                                                  'Lens_Dist','Lens_Teff','Lens_logg','Lens_Fe_H']]
                                

                                                  

,name,event_id,sim_label,t0lens1,tE_ref,u0lens1,rho,alpha,piE,piEE,...,ra_deg,dec_deg,Planet_q,Planet_s,Planet_semimajoraxis,Lens_Mass,Lens_Dist,Lens_Teff,Lens_logg,Lens_Fe_H
18698,RMDC26_306504,306504,RMDC26_1S2L_ML,2.463146e+06,25.844615,0.105249,0.002425,229.560967,0.141494,0.085639,...,267.316513,-29.295535,0.001683,1.079805,1.710213,0.173140,7.907737,3.404932e+03,5.110845e+00,-2.579774e-01
42327,RMDC26_306569,306569,RMDC26_1S2L_ML,2.461476e+06,24.642852,0.693976,0.012609,221.376193,0.232184,-0.031992,...,267.320942,-29.785661,0.000030,1.823651,6.130997,0.145320,7.701172,3.285708e+03,5.151182e+00,-2.235061e-01
65930,RMDC26_306458,306458,RMDC26_1S2L_ML,2.462963e+06,48.772231,-0.686118,0.001695,277.826448,0.057374,0.057372,...,267.633699,-28.282104,0.000508,0.689043,1.475240,0.607599,7.533822,2.000000e-50,2.000000e-50,2.000000e-50
113218,RMDC26_306508,306508,RMDC26_1S2L_ML,2.461831e+06,3.204031,-1.047823,0.004134,8.355477,0.017526,-0.012102,...,266.787238,-29.111641,0.000094,7.345423,6.753009,0.644309,8.187812,5.290116e+03,4.657405e+00,-1.012702e+00
136834,RMDC26_306493,306493,RMDC26_1S2L_ML,2.461475e+06,12.534579,-0.325458,0.002477,299.129988,0.031907,-0.013753,...,268.652470,-28.590173,0.000463,0.872179,1.044982,0.568309,8.014552,3.810028e+03,4.723974e+00,2.532264e-01
184087,RMDC26_306463,306463,RMDC26_1S2L_ML,2.462760e+06,35.724324,0.466056,0.001359,58.499390,0.143057,0.137728,...,267.119864,-29.992989,0.000146,0.785445,1.943682,0.196841,8.085342,2.663881e+03,4.991552e+00,5.109707e-01
184088,RMDC26_306479,306479,RMDC26_1S2L_ML,2.462741e+06,6.242727,-0.124355,0.012551,185.937463,0.069403,0.009660,...,267.773973,-30.306360,0.000008,0.564483,0.810171,0.087532,9.063693,2.000000e-50,2.000000e-50,2.000000e-50
184091,RMDC26_306527,306527,RMDC26_1S2L_ML,2.461638e+06,23.251095,-0.147591,0.001621,117.988258,0.203732,-0.145224,...,268.120227,-29.590190,0.000330,0.791186,1.542934,0.106132,7.277628,2.451298e+03,5.133579e+00,4.454872e-01
254951,RMDC26_306514,306514,RMDC26_1S2L_ML,2.461866e+06,27.950307,0.051585,0.000977,77.521515,0.071418,0.044348,...,268.450787,-29.413837,0.000018,0.921602,3.434816,0.533839,7.183858,2.000000e-50,2.000000e-50,2.000000e-50
278594,RMDC26_306467,306467,RMDC26_1S2L_ML,2.461684e+06,45.085953,0.334545,0.002152,258.070575,0.161055,0.019816,...,268.062339,-28.198232,0.002820,1.178020,1.341146,0.121334,6.589955,3.332798e+03,5.241269e+00,-5.657303e-01


Ok so the concept of a plan of an idea is forming in my simple brain. 

If there is no favored model, the event does not count toward the mass ratio function. However, it can still count toward the total GBTDS planet occurrence rate because it passed Detection, since that is not dependent on s,q. 

If there is only one favored model, I am tempted to forego MCMC, but that goes against the Full Bayesian(TM) Way. So, as long as there's at least one favored model, run MCMC on *every* model, seeding walkers at the RTModel fitted {s,q,etc}. 

Then grab the best-fit {s,q} and see if it matches the injected {s,q} bin. 